# Algorithm Reference — KNN FairRank Family

This notebook is the complete technical reference for every algorithm in the project.
The development narrative and research decisions are in `algorithm_design.ipynb`.

---

## Index

**Theory**

- [1. Problem Statement and the Bias Argument](#problem-statement)
- [2. KNNFairRank — Full Derivation from First Principles](#fairrank-derivation)
- [2.5 v3: Both Ingredients Correct](#v3-complete)
- [3. Optimal Vote Count — Theory](#optimal-vote-count)
- [4. Failure Modes of the Base Algorithm](#failure-modes)

**Implementation Reference — Baseline**

- [5. Implementation Overview — Class Hierarchy](#impl-overview)
- [6. KNNBase — The Lazy Learner Skeleton](#knnbase)
- [7. KNNClassifier and KNNClassifierFast](#knnclassifier)
- [8. KNNOptK — Data-Driven k Selection](#knnoptk)

**Implementation Reference — FairRank Family**

- [9. KNNFairRank — Implementation Walkthrough](#knnfairrank-impl)
- [10. KNNFairRankMagnitude — Modification B](#knnfairrankb)
- [11. KNNFairRankCV — Modification C](#knnfairrankc)
- [12. KNNFairRankMagnitudeCV — Combined B + C](#knnfairrankbc)
- [13. KNNFairRankEnsemble — α-Grid Ensemble](#knnfairrankens)
- [14. KNNFairRankMagnitudeEnsemble — Magnitude + Ensemble](#knnfairrankbens)
- [15. KNNFairRankOptVotes — CV-Tuned Vote Count](#knnfairrankoptvotes)
- [16. KNNFairRankJointCV — Joint CV over n\_votes and α](#knnfairrankjointcv)
- [17. KNNFairRankLocalOdds — Per-Query Local Odds](#knnfairranklocal-odds)
- [18. KNNFairRankLocalCount — Per-Query Minority-Budget Count](#knnfairranklocalcount)
- [19. KNNFairRankBayesian — Bayesian α-Shrinkage](#knnfairrankbayesian)
- [20. KNNFairRankDensityRegion — Dual-Class Density Regions](#knnfairrankdensityregion)
- [21. KNNFairRankJackknife — LOO Jackknife over Minority Ranks](#knnfairrankjackknife)
- [22. KNNFairRankLID — LID-Derived α (No Inner CV)](#knnfairranklid)
- [23. KNNFairRankJackknifeEnsemble — Jackknife + α-Grid Ensemble](#knnfairrankjackknifeens)
- [24. KNNFairRankLocalOddsJackknife — Local Odds + Jackknife](#knnfairranklocaloddsjk)
- [25. KNNFairRankTopoJoint — Topology-Guided Regional Correction](#knnfairranktopojoint)
- [26. KNNFairRankTopoJointBootstrap — Bootstrap-Gated Topology](#knnfairranktopojointbs)
- [27. KNNFairRankTopoCount — Topology-Derived Scale for Counting](#knnfairranktopocount)

<a id="problem-statement"></a>

---
## 1. Problem Statement and the Journey to KNNFairRank

### 1.1 The setting

We study binary classification with class imbalance. Training data consists of $N$ points
$(x_i, y_i)$ where $y_i \in \{0, 1\}$, with class counts $N_\text{maj}$ (majority, label 0)
and $N_\text{min}$ (minority, label 1), and imbalance ratio:

$$r = \frac{N_\text{maj}}{N_\text{min}} \gg 1$$

Standard KNN classifies a query point $x$ by a majority vote over its $k$ nearest training
neighbours. Under class imbalance this vote is systematically biased toward the majority class —
not because $x$ is geometrically close to the majority, but because majority points are simply
more numerous and therefore tend to dominate any neighbourhood.

### 1.2 Two levels of bias

**Surface bias (k-selection).** With large $k$ the neighbourhood swells and is overwhelmed by
majority points. The natural response is to shrink $k$ — and `KNNOptK` does this via inner
cross-validation, selecting the globally optimal $k$ for each dataset. This helps: inner CV
selects $k=1$ in ≈62% of folds in our benchmark, confirming that the optimal global response
is maximal locality. But performance still degrades with IR even at $k=1$, which points to a
deeper problem.

**Structural bias (rank comparison).** Even with $k=1$, standard KNN makes a fundamentally
unfair comparison: it asks whether the *closest minority* neighbour is closer than the *closest
majority* neighbour. With $N_\text{maj} \gg N_\text{min}$, majority points are much denser
in space, so their closest representative is statistically expected to be closer to any query
point — not because of geometry, but because of sampling density. Standard KNN conflates
geometric proximity with class density, and this conflation persists no matter how small $k$ is.

This is the core problem KNNFairRank addresses. Section 2 derives the fix rigorously; the
rest of this section retraces the path that led to it, since understanding the dead ends makes
the fair-rank insight feel inevitable.

<a id="fairrank-derivation"></a>

---
## 2. KNNFairRank — Full Derivation from First Principles

### 2.1 The rank concept

When classifying query point $x$, sort all training points of each class by distance to $x$
separately. Define:

- $d_k^\text{min}(x)$ — distance from $x$ to the $k$-th nearest **minority** training point
- $d_k^\text{maj}(x)$ — distance from $x$ to the $k$-th nearest **majority** training point

"Rank $k$" simply means position $k$ in that sorted list. Rank 1 is the closest, rank 2 the
second closest, and so on. Standard KNN with $k=1$ implicitly compares rank-1 minority against
rank-1 majority and predicts the class whose rank-1 point is closer.

### 2.2 Statistical model of nearest-neighbour distances

Assume training points of class $c$ are drawn from a spatial distribution with local density
$\lambda_c$ (points per unit volume) in a $d$-dimensional feature space. Under a homogeneous
Poisson process approximation, the expected number of class-$c$ points inside a ball of radius
$\rho$ centred at $x$ is:

$$\mathbb{E}[\text{count inside ball}] = \lambda_c \cdot V_d \cdot \rho^d$$

where $V_d = \pi^{d/2} / \Gamma(d/2 + 1)$ is the volume of the unit $d$-ball. The
$k$-th nearest neighbour distance is the smallest $\rho$ such that the expected count reaches
$k$, giving:

$$\mathbb{E}[d_k^c(x)] \propto \left(\frac{k}{\lambda_c}\right)^{1/d}$$

This is the key scaling law. Two observations follow immediately:

1. **Density effect:** for fixed $k$, a denser class ($\lambda_c$ large) has smaller expected
   nearest-neighbour distances. Majority points are denser ($\lambda_\text{maj} > \lambda_\text{min}$)
   so $\mathbb{E}[d_1^\text{maj}] < \mathbb{E}[d_1^\text{min}]$ regardless of $x$.

2. **Rank effect:** increasing $k$ increases the expected distance as $k^{1/d}$.

### 2.3 Quantifying the unfair comparison

Under balanced class priors, $\lambda_\text{maj}/\lambda_\text{min} = N_\text{maj}/N_\text{min} = r$.
Comparing both classes at rank 1:

$$\frac{\mathbb{E}[d_1^\text{min}(x)]}{\mathbb{E}[d_1^\text{maj}(x)]}
= \frac{(1/\lambda_\text{min})^{1/d}}{(1/\lambda_\text{maj})^{1/d}}
= \left(\frac{\lambda_\text{maj}}{\lambda_\text{min}}\right)^{1/d}
= r^{1/d}$$

For $r = 10$ in $d = 2$ dimensions: $r^{1/d} = \sqrt{10} \approx 3.16$. The nearest minority
neighbour is expected to be **3× further away** than the nearest majority neighbour, purely due to
density. This bias grows as $r$ increases and shrinks (but does not vanish) as $d$ increases.

### 2.4 Deriving the fair majority rank

A natural first guess is to use this distance ratio as the rank correction itself: if minority
distances are inflated by a factor of $r^{1/d}$, advance the majority comparison by the same
factor and set $k_\text{eff} = r^{1/d}$. This is what our v1/v2 prototypes did, and it is
wrong. The mistake is conflating ranks with distances: the scaling law shows rank enters
distance space through a $k^{1/d}$ factor, so to *cancel* a distance factor of $r^{1/d}$ the
*rank* factor needs to be $r$, not $r^{1/d}$. The derivation below makes this precise.

A fair comparison requires both sides to have the same expected distance under the null hypothesis
that $x$ lies on the decision boundary (where both classes are equally likely). We want the
majority rank $k_\text{eff}$ such that:

$$\mathbb{E}[d_1^\text{min}(x)] = \mathbb{E}[d_{k_\text{eff}}^\text{maj}(x)]$$

Substituting the scaling law:

$$\left(\frac{1}{\lambda_\text{min}}\right)^{1/d} = \left(\frac{k_\text{eff}}{\lambda_\text{maj}}\right)^{1/d}$$

Raising both sides to the power $d$ (eliminating the exponent):

$$\frac{1}{\lambda_\text{min}} = \frac{k_\text{eff}}{\lambda_\text{maj}}$$

$$\boxed{k_\text{eff} = \frac{\lambda_\text{maj}}{\lambda_\text{min}} = \frac{N_\text{maj}}{N_\text{min}} = r}$$

The $d$-th power eliminated the $1/d$ exponent — the fair rank is $r$, not $r^{1/d}$. The
distance ratio $r^{1/d}$ is the *symptom* of the bias; the rank correction $r$ is the *cure*.

**Decision rule.** KNNFairRank predicts minority iff:

$$d_1^\text{min}(x) < d_r^\text{maj}(x)$$

i.e., the closest minority neighbour is closer than the $r$-th closest majority neighbour.

### 2.5 Multi-rank voting — reducing variance

A single comparison (rank-1 minority vs rank-$r$ majority) is noisy: it depends on whichever
single minority point and single majority point happen to be at those positions, which fluctuates
across query points. To reduce variance, KNNFairRank aggregates multiple independent fair
comparisons.

The same fairness argument scales to higher ranks: if rank-1 minority is fairly compared against
majority rank $r$, then rank-$i$ minority is fairly compared against majority rank $i \cdot r$:

$$\mathbb{E}[d_i^\text{min}(x)] = \left(\frac{i}{\lambda_\text{min}}\right)^{1/d}
= \left(\frac{i \cdot r}{\lambda_\text{maj}}\right)^{1/d}
= \mathbb{E}[d_{i \cdot r}^\text{maj}(x)]$$

For each vote $i = 1, 2, \ldots, n_\text{votes}$, cast:

$$\text{vote}_i =
\begin{cases}
+1 & \text{if } d_i^\text{min}(x) < d_{i \cdot r}^\text{maj}(x) \quad (\text{minority wins}) \\
-1 & \text{otherwise} \quad (\text{majority wins})
\end{cases}$$

Final prediction: minority iff $\sum_{i=1}^{n_\text{votes}} \text{vote}_i > 0$.

Each vote is an independent fair comparison by the same Poisson argument. Aggregating $n_\text{votes}$
comparisons dramatically reduces the sensitivity to any single lucky or unlucky point placement.
In our implementation $n_\text{votes} = 5$.

### 2.6 What the algorithm needs at inference time

For a query point $x$:
1. Compute distances to all training points
2. Sort minority distances → $d_1^\text{min} \leq d_2^\text{min} \leq \ldots$
3. Sort majority distances → $d_1^\text{maj} \leq d_2^\text{maj} \leq \ldots$
4. Compute $r = N_\text{maj}/N_\text{min}$ (stored from training)
5. For $i = 1, \ldots, n_\text{votes}$: compare $d_i^\text{min}$ vs $d_{\lfloor i \cdot r \rfloor}^\text{maj}$
6. Predict minority iff majority of votes favour minority

Note: no fitting is required beyond storing $N_\text{maj}$, $N_\text{min}$, and the training
points. KNNFairRank is a lazy learner like standard KNN.


<a id="v3-complete"></a>

## 2.5 v3 — Both Ingredients Correct

v3 combines the correct $k_\text{eff} = r$ (§2.4) with the multi-rank voting scheme (§2.5).
The two ingredients are independent and each contributes: the fair rank fixes the systematic
bias, the multi-vote averaging reduces variance around it. Removing either one degrades v3 in a
predictable way — to v1 (one fair comparison, noisy) or to v2 (many comparisons, biased).


<a id="optimal-vote-count"></a>

---
## 3. Optimal Vote Count — Theory

The multi-vote scheme raises an immediate question: how many votes is enough, and is there a
point past which adding more votes hurts? This section works out the answer from first
principles.

---

### 3.5.1 All individual votes are unbiased

The rank correction $k_\text{eff} = r$ was motivated in §2 for a single comparison at rank 1.
It is exact in expectation for *every* vote index $i$, not just the first.

By the Poisson-process order-statistics formula:

$$\mathbb{E}[d_k^c] \propto \left(\frac{k}{N_c}\right)^{1/d}$$

Vote $i$ compares $d_i^\text{min}$ against $d_{i \cdot k_\text{eff}}^\text{maj}$. Setting
$k_\text{eff} = r = N_\text{maj}/N_\text{min}$:

$$\mathbb{E}\!\left[d_{i \cdot k_\text{eff}}^\text{maj}\right]
  \propto \left(\frac{i \cdot r}{N_\text{maj}}\right)^{1/d}
  = \left(\frac{i}{N_\text{min}}\right)^{1/d}
  = \mathbb{E}[d_i^\text{min}] \qquad \forall\, i = 1, \ldots, N_\text{min}$$

**The rank correction is self-consistent at every rank.** Adding more votes cannot introduce a
systematic bias — it can only affect variance and correlation structure.

---

### 3.5.2 Votes are positively correlated — the effective-information ceiling

Even though each vote is individually valid, votes are not independent. $d_i^\text{min}$ and
$d_j^\text{min}$ both come from the same sorted distance list: if the query is geometrically
close to the minority cloud, every vote tends to say "minority" simultaneously. Define the
average pairwise inter-vote correlation:

$$\bar{\rho}
  = \frac{1}{\binom{n}{2}} \sum_{i < j} \operatorname{Corr}(V_i,\, V_j),
  \qquad
  V_i = \mathbf{1}\!\left[d_i^\text{min} < d_{i \cdot k_\text{eff}}^\text{maj}\right]$$

For correlated binary votes the **effective number of independent votes** is:

$$n_\text{eff}(n) = \frac{n}{1 + (n-1)\,\bar{\rho}}$$

As $n \to \infty$, $n_\text{eff} \to 1/\bar{\rho}$. Information **saturates** at $1/\bar{\rho}$
regardless of how many votes you cast. For Poisson nearest-neighbour order statistics, adjacent
ranks correlate at $\bar{\rho} \approx 0.2$–$0.4$, so $n_\text{eff,max} \approx 2.5$–$5$.

---

### 3.5.3 Signal degrades with rank

Define the **signal strength** of vote $i$ as:

$$s_i = P\!\left(V_i = \text{minority} \;\middle|\; x \text{ is truly minority}\right) - 0.5 \;\geq 0$$

$s_1$ is the strongest: $d_1^\text{min}$ is the single closest minority point, capturing the
most local geometry. For larger $i$, $d_i^\text{min}$ comes from the $i$-th ring of the
minority cloud — progressively less representative of the query's immediate neighbourhood.
As $i \to \infty$, $s_i \to 0$.

The aggregate fraction $\hat{p}_n = \frac{1}{n}\sum_{i=1}^n V_i$ has expected value
$\frac{1}{n}\sum s_i + 0.5$. Because $s_i$ is decreasing, adding vote $n+1$ lowers the mean
signal of the aggregate once $s_{n+1} < \bar{s}_n$. Beyond the peak, each additional vote
dilutes rather than refines the aggregate.

---

### 3.5.4 Optimal vote count

Two independent upper bounds apply simultaneously:

**Effective-information ceiling** (from §3.5.2):

$$n_\text{votes} \lesssim \frac{1}{\bar{\rho}} \approx 3\text{–}5$$

**Locality principle** (from §3.5.3): the $i$-th nearest minority neighbour lies at distance
$\sim (i/N_\text{min})^{1/d}$. The stable KNN neighbourhood spans roughly
$k^* \sim \sqrt{N_\text{train}}$ points (the empirical optimum under KNNOptK). The comparison
stays local only for $i \lesssim \sqrt{N_\text{min}}$:

$$n_\text{votes} \lesssim \sqrt{N_\text{min}}$$

Taking the binding constraint:

$$\boxed{n^* \approx \min\!\left(\frac{1}{\bar{\rho}},\; \sqrt{N_\text{min}}\right)}$$

For most benchmark datasets $N_\text{min} \geq 20$, so $\sqrt{N_\text{min}} \geq 4.5$ and the
correlation ceiling ($\approx$3–5) dominates. **The default of 5 sits at the theoretical
optimum for the bulk of the benchmark suite.**

| $N_\text{min}$ | $\sqrt{N_\text{min}}$ | $1/\bar{\rho}$ (typical) | $n^*$ |
|:---:|:---:|:---:|:---:|
| 10 | 3.2 | 3–5 | **3** |
| 25 | 5.0 | 3–5 | **3–5** |
| 50 | 7.1 | 3–5 | **3–5** |
| 100 | 10 | 3–5 | **3–5** |
| 400 | 20 | 3–5 | **3–5** |

---

### 3.5.5 High-IR pathology

At large imbalance ratio $r$, vote $n$ reaches majority rank $n \cdot r$. For
$n_\text{votes} = 5$ and $r = 100$, vote 5 compares $d_5^\text{min}$ with $d_{500}^\text{maj}$
— the 500th nearest majority point, well outside any local neighbourhood. A tighter locality
constraint for high-IR datasets is:

$$n_\text{votes} \leq \left\lfloor \frac{\sqrt{N_\text{maj}}}{r} \right\rfloor
  = \left\lfloor \frac{N_\text{min}}{\sqrt{N_\text{maj}}} \right\rfloor$$

For $N_\text{min} = 20$, $N_\text{maj} = 2000$ ($r = 100$): bound
$= \lfloor 20 / \sqrt{2000} \rfloor = 0$. Even a single vote reaches far into the majority
cloud in this extreme regime. In practice this is mitigated by `k_maj_cap` (§16), which
prevents the majority neighbourhood buffer from growing without bound — but it is worth knowing
that the *conceptual* quality of each comparison degrades steeply as $r$ grows.

---

### 3.5.6 Empirical findings — two regimes, not one

Running the sweep over the full benchmark (45 datasets, 5-fold CV) produces a
**bimodal** distribution of per-dataset optimal $n_\text{votes}$:

| $n_\text{votes}$ | Datasets at peak |
|:---:|:---:|
| 1 | 10 |
| 2 | 18 |
| 3 | 2 |
| 5 | 3 |
| 7 | 4 |
| 10 | 8 |

Two distinct regimes emerge with almost nothing in between:

- **Few-votes regime** ($n \in \{1, 2\}$, 28 datasets): small $N_\text{min}$ or
  tightly-clustered minorities where ranks 1–2 capture all useful local geometry.
  Consistent with the locality bound $n^* \lesssim \sqrt{N_\text{min}}$ being small.
- **Many-votes regime** ($n \in \{7, 10\}$, 12 datasets): larger or more diffuse
  minority clouds where the locality bound is not the binding constraint and
  additional votes genuinely reduce variance.

The fixed default $n_\text{votes} = 5$ falls in the dead zone between the two
clusters: only **7 / 45 datasets** are within 0.5% of optimal at $n=5$.

**Why does the aggregate G-mean still grow past $n=2$?**
Datasets in the few-votes regime *plateau* after their peak rather than declining —
extra votes carry near-zero signal, not negative signal, so they do not pull the
aggregate down. Datasets in the many-votes regime continue improving and push the
aggregate curve up. The net result is a monotonically increasing aggregate even
though most datasets peaked long before.

---

### 3.5.7 CV-selected vote count — `KNNFairRankOptVotes`

Since no fixed $n_\text{votes}$ covers both regimes, the natural fix is to select it
via inner CV, exactly as `KNNFairRankCV` selects $\alpha$.

**Why inner CV is cheap here:** the expensive step in each query is
`_per_class_distances` (the `cdist` call), which depends only on `k_min` and
`k_maj` — both set at `fit` time and independent of $n_\text{votes}$. Evaluating
different vote counts on pre-fitted neighbours amounts to array indexing into
already-sorted distances. The inner CV therefore pays only for extra `fit()` calls,
not extra distance computations — the same class of overhead as `KNNFairRankCV`.

**Implementation (`knn_fair_rank_opt_votes.py`):** selects $n_\text{votes}$ from the
grid $\{1, 2, 3, 5, 7, 10\}$ via 3-fold stratified inner CV optimising G-mean.
The neighbourhood is sized for the largest grid candidate so all values can be
evaluated within a single outer `fit()`. The selected count is stored in
`best_n_votes_` for inspection.

In [ ]:
import sys
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedKFold

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.loader import load_all_datasets
from src.data.preprocessing import binarise_labels, remove_constant_features, standardise
from src.algorithms.fair_rank.core import KNNFairRank
from src.evaluation.metrics import geometric_mean

# ─── Configuration ────────────────────────────────────────────────────────────
N_VOTES_GRID = [1, 2, 3, 5, 7, 10, 15]
N_FOLDS      = 5
MIN_N_MIN    = 10      # skip datasets too small for 5-fold CV
SEED         = 42
N_JOBS       = 4       # dataset-level parallelism; set to 1 to disable
CACHE        = project_root / "results" / "tables" / "nvotes_sweep.csv"

# ─── Worker: one dataset, all n_votes ─────────────────────────────────────────
def _sweep_dataset(ds, n_votes_grid, n_folds, seed, min_n_min):
    import warnings, numpy as np
    from src.data.preprocessing import binarise_labels, remove_constant_features, standardise
    from src.algorithms.fair_rank.core import KNNFairRank
    from src.evaluation.metrics import geometric_mean
    from sklearn.model_selection import StratifiedKFold

    X = remove_constant_features(ds.X)
    y = binarise_labels(ds.y)
    n_min = int(y.sum())
    if n_min < min_n_min:
        return []

    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    records = []
    for nv in n_votes_grid:
        fold_scores = []
        for tr, va in cv.split(X, y):
            X_tr, X_va = standardise(X[tr], X[va])
            clf = KNNFairRank(n_votes=nv)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                clf.fit(X_tr, y[tr])
                fold_scores.append(geometric_mean(y[va], clf.predict(X_va)))
        records.append({
            "dataset":  ds.name,
            "n_min":    n_min,
            "ir_ratio": ds.imbalance_ratio,
            "n_votes":  nv,
            "gmean":    float(np.mean(fold_scores)),
        })
    return records

# ─── Load datasets ────────────────────────────────────────────────────────────
datasets = load_all_datasets()
print(f"Loaded {len(datasets)} datasets.")

# ─── Run or resume from cache ─────────────────────────────────────────────────
if CACHE.exists():
    df = pd.read_csv(CACHE)
    print(f"Loaded from cache ({df['dataset'].nunique()} datasets, delete {CACHE.name} to re-run).")
else:
    _cpu = os.cpu_count() or 4
    print(f"CPUs: {_cpu}  |  N_JOBS={N_JOBS}")

    all_records = []
    gen = Parallel(n_jobs=N_JOBS, prefer="threads", return_as="generator_unordered")(
        delayed(_sweep_dataset)(ds, N_VOTES_GRID, N_FOLDS, SEED, MIN_N_MIN)
        for ds in datasets
    )
    for i, records in enumerate(gen):
        if not records:
            continue
        all_records.extend(records)
        ds_name = records[0]["dataset"]
        n_min   = records[0]["n_min"]
        print(f"  [{i+1:>2}/{len(datasets)}] {ds_name}  (n_min={n_min})")

    df = pd.DataFrame(all_records)
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(CACHE, index=False)
    print(f"\nSaved to {CACHE.name}  ({df['dataset'].nunique()} datasets included).")

# ─── Figure 1: overall median G-mean ± IQR ───────────────────────────────────
agg = df.groupby("n_votes")["gmean"].agg(
    median="median",
    q25=lambda x: x.quantile(0.25),
    q75=lambda x: x.quantile(0.75),
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.fill_between(agg.index, agg["q25"], agg["q75"], alpha=0.2, color="#2878B5")
ax.plot(agg.index, agg["median"], "o-", color="#2878B5", lw=2, label="median G-mean")
ax.axvline(5, color="grey", lw=1.2, ls="--", label="default (n=5)")
ax.axvspan(1/0.4, 1/0.2, alpha=0.12, color="orange",
           label=r"$1/\bar{\rho}$ range ($\bar{\rho}=0.2$–$0.4$)")
ax.set_xlabel("n_votes")
ax.set_ylabel("G-mean")
ax.set_title("Overall — median ± IQR across all datasets")
ax.legend(fontsize=9)
ax.set_xticks(N_VOTES_GRID)

# ─── Figure 2: by IR quartile ────────────────────────────────────────────────
df["ir_quartile"] = pd.qcut(
    df["ir_ratio"], q=4,
    labels=["Q1 (most severe IR)", "Q2", "Q3", "Q4 (mildest IR)"]
)
colors = ["#C0392B", "#E67E22", "#27AE60", "#2878B5"]

ax = axes[1]
for label, color in zip(["Q1 (most severe IR)", "Q2", "Q3", "Q4 (mildest IR)"], colors):
    sub = df[df["ir_quartile"] == label].groupby("n_votes")["gmean"].median()
    ax.plot(sub.index, sub.values, "o-", color=color, lw=2, label=label)
ax.axvline(5, color="grey", lw=1.2, ls="--")
ax.set_xlabel("n_votes")
ax.set_ylabel("G-mean")
ax.set_title("By IR quartile — median G-mean")
ax.legend(fontsize=9)
ax.set_xticks(N_VOTES_GRID)

plt.tight_layout()
plt.show()

# ─── Figure 3: distribution of per-dataset optimal n_votes ───────────────────
pivot   = df.pivot_table(index="dataset", columns="n_votes", values="gmean")
peak_nv = pivot.idxmax(axis=1)
counts  = peak_nv.value_counts().reindex(N_VOTES_GRID, fill_value=0)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar([str(v) for v in counts.index], counts.values,
       color="#2878B5", edgecolor="white", width=0.6)
ax.axvline(counts.index.tolist().index(5), color="grey", lw=1.2, ls="--",
           label="default (n=5)")
ax.set_xlabel("n_votes at peak G-mean")
ax.set_ylabel("Number of datasets")
ax.set_title("Distribution of per-dataset optimal n_votes")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# ─── Summary ─────────────────────────────────────────────────────────────────
print("\nPer-dataset peak n_votes:")
print(counts.to_string())
best  = pivot.max(axis=1)
at5   = pivot[5]
close = ((best - at5) / best.clip(lower=1e-9) < 0.005).sum()
print(f"\nDatasets where n=5 is optimal or within 0.5% of optimal: {close} / {len(pivot)}")

<a id="failure-modes"></a>

---
## 4. Where the Poisson Approximation Breaks — Failure Modes of v3

The fair-rank derivation in §2 assumes training points are drawn from a homogeneous Poisson
process: uniform spatial density, independence, no clustering. Real datasets only ever
*approximately* satisfy this. The interesting question is not whether the assumption holds (it
doesn't) but where it breaks and how each break affects v3's predictions.

Three of the four failure modes below — F1, F2, F4 — are different manifestations of the same
underlying issue: the Poisson-uniform model is only an approximation. F3 is genuinely separate
and concerns how votes are *combined* once they have been cast. Together they motivate
Modifications A–E in §§5–9.

### 4.1 F1 — Local density is non-uniform (Poisson approximation, version 1)

The Poisson-uniform model predicts class-$c$ density is constant: $\lambda_c$ everywhere.
Real minority classes typically cluster — there are pockets of high local minority density
separated by gaps with almost no minority points. Inside a cluster the local minority density
exceeds the global average; far from any cluster it falls well below.

Because v3 uses the *global* $r$ everywhere, the same correction is applied to query points in
both regimes. Near a minority cluster the correction over-shoots (local $r_\text{local} < r$);
far from any cluster it under-shoots. The error has opposite sign in the two regimes, which
shows up as high dataset-to-dataset variance in v3's performance.

### 4.2 F2 — Global $r$ is the wrong target locally (Poisson approximation, version 2)

F2 is the same kind of mismatch as F1 but on the majority side. Even when minority points
*are* roughly uniform, the majority class often is not — it has dense regions and sparse
regions of its own. Near a majority-dense region, $r_\text{local} > r$ and a stronger
correction is needed; near a majority-sparse region, less is needed. v3's single global
$k_\text{eff}$ cannot adapt to either.

### 4.3 F3 — Binary votes discard margin

This failure mode does not depend on the Poisson approximation at all — it concerns how votes
are aggregated. The binary vote $\text{vote}_i \in \{+1, -1\}$ treats a comparison where
$d_i^\text{min} = 0.001$ and $d_{ir}^\text{maj} = 100$ identically to one where
$d_i^\text{min} = 0.49$ and $d_{ir}^\text{maj} = 0.51$. The former is overwhelming evidence
for minority; the latter is a coin flip. Aggregating binary votes throws away that distinction,
so the resulting vote fraction is a poor proxy for classification confidence — which matters
for `predict_proba` and threshold tuning even when it does not change hard-label decisions.

### 4.4 F4 — No dampening when the approximation fails globally (Poisson approximation, version 3)

F4 is the third Poisson-approximation manifestation. On datasets where the Poisson-uniform
assumption is *globally* violated — for example, a heavily clustered minority class in a
low-dimensional space — v3 may systematically over-correct everywhere. There is no parameter
to dial the correction down. The only tool v3 has is its derived value $k_\text{eff} = r$,
which is correct under the assumption but not when the assumption is wrong.

The effect is that on hostile datasets v3's G-mean can drop below SMOTE+KNN or even KNNOptK.
What is missing is an escape hatch — a way to interpolate between "trust the theory fully"
and "trust the data more than the theory".


<a id="impl-overview"></a>

---
## 5. Implementation Overview — Class Hierarchy and Design Principles

### 12.1 Class hierarchy

```
KNNBase                           src/algorithms/baseline/knn_base.py
└── KNNClassifier                 src/algorithms/baseline/knn_base.py
    ├── KNNClassifierFast         src/algorithms/baseline/knn_base.py
    └── KNNFairRank               src/algorithms/fair_rank/core/knn_fair_rank.py
        ├── KNNFairRankMagnitude  src/algorithms/fair_rank/core/knn_fair_rank_b.py
        └── KNNFairRankCV         src/algorithms/fair_rank/core/knn_fair_rank_c.py

KNNOptK          (standalone)    src/algorithms/baseline/knn_base.py
KNNAdaptiveTopo  (standalone)    src/algorithms/adaptive_k/knn_adaptive_topo.py
```

`KNNFairRankMagnitude` and `KNNFairRankCV` are **parallel modifications** — they extend `KNNFairRank` independently, not each other.

`KNNOptK` stands outside the hierarchy because it is a model-selection wrapper, not a classifier. It internally owns a private `KNNClassifierFast` instance and delegates all predictions to it.

### 12.2 What each level of the hierarchy contributes

| Class | What it adds |
|---|---|
| `KNNBase` | `fit` (store training data), `predict`/`predict_proba` dispatch with optional parallelism, reference `_predict_x` |
| `KNNClassifier` | `aggregate` — majority vote over $k$ neighbour labels |
| `KNNClassifierFast` | Vectorised `_predict_x`/`_predict_proba_x` using `cdist` (bit-for-bit equal to `KNNClassifier`, ~10–50× faster) |
| `KNNFairRank` | Separate per-class distance lists, rank-corrected multi-vote decision rule; overrides both `_predict_x` and `_predict_proba_x` completely |
| `KNNFairRankMagnitude` | Overrides `_vote_fraction` to replace binary vote with continuous score |
| `KNNFairRankCV` | Overrides `fit` to select α by inner CV; overrides `_vote_fraction` to use `k_eff = r^α` |

### 12.3 The override pattern

Every algorithm follows the same pattern:

1. Call `super().__init__(...)` to set up shared infrastructure (`n_jobs`, `k`)
2. Call `super().fit(X, y)` to store `self.X`, `self.y`, `self.classes_`
3. Override `_predict_x` and/or `_predict_proba_x` to change the decision rule
4. In FairRank variants, override `_vote_fraction` to change how votes are computed within that rule

The parallel dispatch code in `KNNBase.predict` is written once and inherited by every classifier. A subclass that overrides `_predict_x` automatically gets parallelism with no additional code.

### 12.4 The k=1 dummy in KNNFairRank

`KNNFairRank.__init__` calls `super().__init__(k=1, n_jobs=n_jobs)`. The value `k=1` is a placeholder. `KNNFairRank` overrides **both** `_predict_x` and `_predict_proba_x`, and neither references `self.k`. The `super().__init__` call is needed only to set `self.n_jobs` for the dispatch layer. `k=1` was chosen as the value that would make the base `_predict_x` behave as standard 1-NN if the overrides were somehow bypassed — but it has no effect in practice.

### 12.5 Configuration system

All hyperparameter defaults live in `config/settings.yaml`. Each class reads its section at construction time:

```python
cfg = load_config().get("knn_fair_rank", {})
self._k_min = k_min if k_min is not None else cfg.get("k_min", 10)
```

Priority: **explicit constructor argument > config file value > hardcoded fallback**.

This means:
- Benchmarking code instantiates models with no arguments; the config file controls all hyperparameters
- Tests pass explicit arguments to override the config for controlled scenarios
- The config file is the single source of truth for all values used in experiments

### 12.6 scikit-learn interface compatibility

All classes expose `fit(X, y)`, `predict(X)`, and `predict_proba(X)` with NumPy array inputs and outputs. No `sklearn.base.BaseEstimator` inheritance is declared, but the interface is duck-typed to be compatible with `StratifiedKFold`, `cross_val_score`, and similar scikit-learn utilities. The `classes_` attribute set in `fit` follows the scikit-learn convention used by `predict_proba` callers.

<a id="knnbase"></a>

---
## 6. KNNBase — The Lazy Learner Skeleton

`KNNBase` is defined in `src/algorithms/baseline/knn_base.py` and provides the minimal shared infrastructure for all KNN variants. It is never instantiated directly.

### 13.1 Constructor

```python
def __init__(self, k=5, distance_func=euclidean, n_jobs=1):
    self.k = None if k == 0 else k
    self.distance_func = distance_func
    self.n_jobs = n_jobs
```

- **`k = 0` sentinel.** Setting `self.k = None` means "use all training examples." Python slice semantics make `lst[:None]` equivalent to `lst[:]` (the full list), so this propagates correctly through `_predict_x` without any special-casing.
- **`distance_func`** defaults to `scipy.spatial.distance.euclidean`. It is only used in the reference `_predict_x`; subclasses that override `_predict_x` (all of them) effectively ignore this parameter.
- **`n_jobs`** controls the number of parallel workers in `predict` and `predict_proba`.

### 13.2 `fit` — lazy storage

```python
def fit(self, X, y):
    self.X = np.asarray(X, dtype=float)
    self.y = np.asarray(y)
    self.classes_ = np.unique(self.y)
    return self
```

`fit` does nothing except store the training data and compute the sorted unique label array. This is the defining property of a lazy learner: training is $O(N)$ (memory copy), with all computation deferred to prediction time.

`np.asarray(X, dtype=float)` converts the input to a float64 NumPy array. If `X` is already float64, this is a zero-copy view. The conversion ensures distance computations work regardless of whether the caller passes integers, booleans, or categorical encodings.

`self.classes_` follows the scikit-learn convention of a sorted array of unique labels. It is used in `predict_proba` to assign probabilities to the correct column index without hardcoding class values.

### 13.3 `predict` and `predict_proba` — dispatch

```python
def predict(self, X):
    X = np.asarray(X, dtype=float)
    if self.n_jobs == 1:
        return np.array([self._predict_x(x) for x in X])
    return np.array(
        Parallel(n_jobs=self.n_jobs, prefer="threads")(
            delayed(self._predict_x)(x) for x in X
        )
    )
```

The dispatch layer handles exactly one concern: parallelism. All prediction logic lives in `_predict_x`. Subclasses override `_predict_x` to change the decision rule; they get parallel dispatch for free with no additional code.

**Thread-based parallelism.** `prefer="threads"` uses OS threads (not processes). Threads share memory — no data copying between workers — which avoids the overhead of pickling the training matrix `self.X` for each worker. The Python GIL would normally block true parallelism, but `cdist` (used in all fast subclasses) releases the GIL during its C-level computation, so threads genuinely run concurrently during distance calculation.

### 13.4 `_predict_x` — reference implementation

```
PROCEDURE KNNBase._predict_x(x)
  Input:  x ∈ ℝ^d, a single query point
  Output: predicted class label

  1: distances ← (distance_func(x, xᵢ) for xᵢ in self.X)   // lazy generator
  2: neighbors ← sort_by_distance(zip(distances, self.y))    // O(N log N)
  3: top_k_labels ← [label for (_, label) in neighbors[:self.k]]
  4: return aggregate(top_k_labels)
```

The generator on line 1 is lazy — distances are computed one at a time as `sorted()` consumes the generator. This avoids materialising all $N$ distances into a list before sorting, saving $O(N)$ extra memory.

**`self.k = None`** causes `neighbors[:None]` to return all neighbours (Python standard). The result of `aggregate` on the full label array is the majority class across all training points, which is the simplest possible classifier.

### 13.5 `aggregate` — abstract method

```python
def aggregate(self, neighbors_targets):
    raise NotImplementedError()
```

`aggregate` receives a list of $k$ labels (the targets of the $k$ nearest training points) and returns a single prediction. It is the one method subclasses must implement. The rest of the prediction pipeline — distance computation, sorting, top-$k$ selection, parallelism — is inherited and shared.

<a id="knnclassifier"></a>

---
## 7. KNNClassifier and KNNClassifierFast

Both classes are defined in `src/algorithms/baseline/knn_base.py`. `KNNClassifier` makes `KNNBase` concrete by implementing `aggregate`; `KNNClassifierFast` replaces the distance-computation loop with a vectorised equivalent while keeping predictions bit-for-bit identical.

### 14.1 KNNClassifier — the reference implementation

`KNNClassifier` adds a single method:

```python
def aggregate(self, neighbors_targets):
    return Counter(neighbors_targets).most_common(1)[0][0]
```

`Counter(neighbors_targets)` builds a frequency table of the $k$ neighbour labels in $O(k)$. `.most_common(1)` returns the single most frequent label in $O(|\text{classes}|)$. The result is a plain Python scalar.

**Tie-breaking.** When two classes share the same count (possible with even $k$), `Counter.most_common` returns one of them in an unspecified order — determined by Python's dictionary hash iteration, which is non-deterministic across runs. This is why `KNNOptK` restricts its search to odd $k$ values (see §15.2).

`KNNClassifier` inherits `_predict_x` from `KNNBase` unchanged. That method computes all $N$ distances in a Python generator loop, sorts them, takes the top-$k$, and calls `aggregate`. It is slow but correct and easy to read.

### 14.2 KNNClassifierFast — vectorised distances

`KNNClassifierFast` overrides `_predict_x` and `_predict_proba_x`:

```python
def _predict_x(self, x):
    dists = cdist(self.X, x.reshape(1, -1), metric='euclidean').ravel()
    idx = np.argsort(dists)[: self.k]
    return self.aggregate(self.y[idx].tolist())
```

The only difference from `KNNBase._predict_x` is how distances are computed:
- `KNNBase`: Python generator `(euclidean(x, example) for example in self.X)` — one Python function call per training point
- `KNNClassifierFast`: `cdist(self.X, x.reshape(1,-1))` — one C-level BLAS call for all $N$ distances simultaneously

The mathematical formula is identical in both cases (Euclidean distance), so the outputs are bit-for-bit equal. The speedup comes entirely from eliminating the Python per-point loop: `cdist` dispatches to optimised LAPACK routines that operate on the full matrix in one call, typically 10–50× faster than the loop.

**`argsort` vs `argpartition`.** `KNNClassifierFast` uses `np.argsort(dists)[:k]` — a full $O(N \log N)$ sort. `KNNFairRank` uses `argpartition + sort` for $O(N + k \log k)$. The reason `KNNClassifierFast` does not use `argpartition` is that it is the **baseline** — simplicity and readability are preferred, and `KNNClassifierFast` is only used inside `KNNOptK`, where the dominant cost is fitting over many inner CV folds, not the per-query sort.

### 14.3 Why KNNClassifier still exists

`KNNClassifier` is the original implementation from the upstream code base (`rushter/MLAlgorithms`). It is kept for two reasons:
1. **Correctness reference.** Any change to `KNNClassifierFast` can be validated by comparing against `KNNClassifier` on a small dataset.
2. **Metric generality.** `KNNBase._predict_x` accepts an arbitrary `distance_func` parameter. `KNNClassifier` inherits this and can be used with any distance metric (Manhattan, Minkowski, etc.). `KNNClassifierFast` is hardcoded to Euclidean via `cdist`.

In all benchmarks and experiments, `KNNClassifierFast` is used exclusively. `KNNClassifier` appears only in tests.

<a id="knnoptk"></a>

---
## 8. KNNOptK — Data-Driven k Selection

`KNNOptK` is defined in `src/algorithms/baseline/knn_base.py`. It is not itself a KNN classifier — it is a **model-selection wrapper** that chooses the best $k$ by inner stratified cross-validation and then delegates all prediction to a private `KNNClassifierFast` instance.

### 15.1 Architecture

`KNNOptK` does not inherit from `KNNBase`. It exposes the same `fit`/`predict`/`predict_proba` interface but is structured as a wrapper:

```
KNNOptK
├── fit(X, y):  run inner CV → select best_k → train self._clf = KNNClassifierFast(k=best_k)
├── predict(X): return self._clf.predict(X)
└── predict_proba(X): return self._clf.predict_proba(X)
```

After fitting, `KNNOptK` is a thin shell around `self._clf`. All predictions are forwarded directly.

### 15.2 `fit` — the k-selection procedure

```
PROCEDURE KNNOptK.fit(X, y)
  Input:  X ∈ ℝ^{N×d}, y ∈ {0,1}^N
  Output: fitted model; self.best_k_ set; self._clf ready for prediction

  // Build the search grid
  1: k_upper ← k_max if k_max is not None else floor(sqrt(N))
  2: k_range ← [k : k ∈ {1, 3, 5, ..., k_upper}]      // odd values only

  // Prepare inner CV
  3: cv ← StratifiedKFold(inner_cv_folds, shuffle=True, seed)
  4: splits ← [(X[tr], X[va], y[tr], y[va]) for (tr, va) in cv.split(X, y)]

  // Evaluate every (split, k) combination in parallel
  5: jobs ← {(X_tr, X_va, y_tr, y_va, k) : for each split, for each k in k_range}
  6: results ← Parallel(n_jobs)(
         for each job (X_tr, X_va, y_tr, y_va, k):
             if k ≥ |X_tr|: return (k, None)              // skip: k ≥ training size
             clf_k ← KNNClassifierFast(k=k)
             clf_k.fit(X_tr, y_tr)
             score ← balanced_accuracy(y_va, clf_k.predict(X_va))
             return (k, score)
     )

  // Aggregate per-k scores across folds
  7: k_scores ← {k: [s for (k', s) in results if k' = k and s is not None]}
  8: valid ← {k: mean(k_scores[k]) : |k_scores[k]| > 0}

  // Select best k and fit final model
  9:  best_k ← argmax_k valid[k]
 10:  self.best_k_ ← best_k
 11:  self.k_range_ ← k_range           // expose for diagnostic inspection
 12:  self._clf ← KNNClassifierFast(k=best_k)
 13:  self._clf.fit(X, y)               // fit on full training data, not just one fold
```

**Line 1 — automatic k range.** `floor(sqrt(N))` is a standard heuristic upper bound: it grows slowly with dataset size and keeps the search tractable. For $N = 900$ training points, $k_\text{upper} = 30$, giving 15 candidate values. `k_max` in the config file overrides this.

**Line 2 — odd values only.** With an even $k$, a 50/50 class split in the neighbourhood produces a tie which `Counter.most_common` resolves arbitrarily. Restricting to odd $k$ guarantees a strict majority winner every time.

**Line 3 — `StratifiedKFold`.** Preserves the class distribution in every fold. On an imbalanced dataset, a plain `KFold` could accidentally create a validation fold with zero minority samples, which would make `balanced_accuracy` undefined or misleading.

**Line 4 — pre-computing splits.** The splits are computed once and reused for all $k$ values. This avoids regenerating the same random fold boundaries once per $k$.

**Line 5 — the job list.** All `|splits| × |k_range|` combinations are independent and can run in any order. They are collected into a flat list and dispatched to `Parallel`.

**Line 6 — parallel execution.** `joblib.Parallel` with `prefer="threads"` runs the evaluations concurrently. Each inner job fits and predicts on a fraction of the training data, so the dominant cost is the $O(N^2 d / \text{folds})$ prediction step.

**The `k ≥ |X_tr|` guard** (inner `if`). KNN with $k \geq$ training size would include every training point as a neighbour — not meaningful. These jobs return `None` and are filtered out when aggregating scores.

**Lines 12–13 — final fit on full data.** After selecting `best_k`, a new classifier is fit on all of `X` (not just the training folds). Cross-validation is used only for model selection; the final model must see all available data.

### 15.3 Scoring metric: balanced accuracy

The inner CV evaluates each $k$ using `balanced_accuracy_score`. Balanced accuracy is the arithmetic mean of sensitivity and specificity:

$$\text{BA} = \frac{1}{2}\left(\frac{TP}{TP+FN} + \frac{TN}{TN+FP}\right)$$

Plain accuracy on an imbalanced dataset would favour large $k$ (the neighbourhood fills with majority points, so predicting majority always is rewarded). Balanced accuracy gives equal weight to both classes, so it penalises the majority-default strategy and rewards classifiers that identify minority points.

### 15.4 Complexity

- **Inner CV**: `inner_cv_folds × |k_range|` fits of `KNNClassifierFast`, each on $(1 - 1/\text{folds})N$ points. Each predict call on a validation fold of size $N/\text{folds}$ costs $O((N/\text{folds}) \cdot N \cdot d)$. Total CV cost: $O(|k_\text{range}| \cdot N^2 \cdot d)$.
- **Final fit**: $O(N)$.
- **Prediction**: $O(M \cdot N \cdot d)$ for a test set of $M$ points — identical to `KNNClassifierFast`.

The inner CV is the bottleneck and scales quadratically in $N$, which matches standard KNN. The `|k_range|` factor (typically 10–30) is the extra cost of model selection.

<a id="knnfairrank-impl"></a>

---
## 9. KNNFairRank — Implementation Walkthrough

`KNNFairRank` is the core algorithm, defined in `src/algorithms/fair_rank/core/knn_fair_rank.py`. This section walks through every method in execution order, connecting each implementation detail to the theory in Section 2.

### 16.1 Constructor parameters

```python
KNNFairRank(
    k_min=10,           # minority neighbourhood size (upper bound on n_votes)
    k_maj_buffer=10,    # additive safety margin when sizing majority neighbourhood
    k_maj_floor=30,     # lower bound on majority neighbourhood size
    k_maj_cap=1000,     # upper bound on majority neighbourhood size
    n_votes=5,          # maximum number of rank-corrected comparisons per query
)
```

Defaults are read from `config/settings.yaml`. Explicit constructor arguments take precedence.

- **`k_min`**: the number of minority training points to fetch distances to for each query. It is an upper bound on how many votes can be cast (vote $i$ needs $d_i^\text{min}$, so you can cast at most `k_min` votes). Default 10 allows up to 10 independent comparisons.
- **`k_maj_buffer`**: additive safety added to `ceil(r) * n_votes` when computing the majority neighbourhood size. Ensures the neighbourhood slightly exceeds the worst-case demand even after integer rounding.
- **`k_maj_floor`**: lower bound on the majority neighbourhood. On nearly-balanced datasets ($r \approx 1$), `ceil(r) * n_votes` could be as small as 5, which is too few points for stable distances. The floor keeps the neighbourhood at a sensible minimum.
- **`k_maj_cap`**: upper bound. At extreme imbalance ($r = 200$, $n_\text{votes} = 5$), uncapped sizing gives `200 * 5 + 10 = 1010`. On a dataset with 500 majority training points that would just clip to 500. The cap avoids allocating unnecessarily large neighbourhood arrays.
- **`n_votes`**: how many fair comparisons to aggregate per query. More votes reduce variance but require larger neighbourhoods and more distance fetches.

The constructor ends with `super().__init__(k=1, n_jobs=n_jobs)`. The `k=1` is a dummy — `KNNFairRank` overrides both `_predict_x` and `_predict_proba_x` and neither references `self.k`. The `super().__init__` call exists only to set `self.n_jobs`, which `KNNBase.predict` uses for parallel dispatch.

### 16.2 `fit` — storing training state

```
PROCEDURE KNNFairRank.fit(X, y)
  Input:  X ∈ ℝ^{N×d}, y ∈ {0,1}^N
  Output: fitted model

  1:  super().fit(X, y)               // stores self.X, self.y, self.classes_
  2:  counts ← {c: |{i : y_i = c}| for c in classes_}
  3:  minority_class ← argmin_c counts[c]
  4:  majority_class ← argmax_c counts[c]
  5:  N_min ← counts[minority_class]
  6:  N_maj ← counts[majority_class]
  7:  self._r ← N_maj / max(1, N_min)             // global imbalance ratio
  8:  self._X_min ← X[y == minority_class]        // minority training points
  9:  self._X_maj ← X[y == majority_class]        // majority training points
 10:  k_maj ← max(ceil(r) * n_votes + k_maj_buffer, k_maj_floor)
 11:  if k_maj_cap is not None: k_maj ← min(k_maj, k_maj_cap)
 12:  self._k_maj_eff ← min(k_maj, |X_maj|)       // cannot exceed available majority points
 13:  self._k_min_eff ← min(k_min, |X_min|)       // cannot exceed available minority points
```

**Line 7.** `max(1, N_min)` prevents division by zero if a degenerate dataset has no minority samples in the training fold. In that case `_r = N_maj`, a large but finite number.

**Line 10 — neighbourhood sizing formula.** In the worst case, all `n_votes` votes are cast and each uses the full `k_eff = ceil(r)` majority rank. The highest majority rank accessed is `n_votes * ceil(r)`. The neighbourhood must be at least this large. `k_maj_buffer` adds a small safety margin. `k_maj_floor` provides a lower bound for nearly-balanced datasets.

**What is NOT done at fit time.** No distances are computed. No tree structures (kd-tree, ball tree) are built. KNNFairRank is a pure **lazy learner**: the only fit-time work is counting, splitting, and computing two integers. All distance computation is deferred to prediction time.

### 16.3 `_per_class_distances` — fetching sorted neighbourhoods

```
PROCEDURE _per_class_distances(x)
  Input:  x ∈ ℝ^d
  Output: (d_min, d_maj) — sorted distance arrays of size k_min_eff and k_maj_eff

  1: d_min_all ← cdist(X_min, x.reshape(1,-1), 'euclidean').ravel()   // ℝ^{|X_min|}
  2: d_maj_all ← cdist(X_maj, x.reshape(1,-1), 'euclidean').ravel()   // ℝ^{|X_maj|}

  // Efficient top-k extraction
  3: if k_min_eff < |d_min_all|:
  4:     idx ← argpartition(d_min_all, k_min_eff - 1)[:k_min_eff]     // O(N_min)
  5:     d_min ← sort(d_min_all[idx])                                   // O(k_min_eff log k_min_eff)
  6: else: d_min ← sort(d_min_all)

  7: (repeat lines 3–6 for majority)
  8: return d_min, d_maj
```

**`cdist` on per-class matrices.** Distances are computed against `_X_min` and `_X_maj` separately, not against the full `self.X`. This is correct and intentional: KNNFairRank maintains independent ranked lists for each class. Using the full training matrix and then splitting would be wasteful because the majority neighbourhood is much larger than the minority neighbourhood.

**`argpartition` vs `argsort`.** A full sort is $O(N \log N)$. `np.argpartition(arr, k)` uses introselect to rearrange the array such that index $k$ holds the $(k{+}1)$-th smallest element and all elements before it are ≤ it — in $O(N)$ time. We then sort only the top-$k$ subset in $O(k \log k)$. Total: $O(N + k \log k)$ vs $O(N \log N)$. For `k_maj_eff = 30` against `N_maj = 1000` majority points this is approximately 3× faster than a full sort.

**When `k_eff ≥ len(d_all)`.** The `if` branch is skipped and all distances are sorted. The clips applied in `fit` (lines 12–13) guarantee `k_min_eff ≤ |X_min|` and `k_maj_eff ≤ |X_maj|`, so the `argpartition` index is always valid.

### 16.4 `_estimate_lid` — local intrinsic dimensionality

```
PROCEDURE _estimate_lid(distances)
  Input:  sorted array of k nearest-neighbour distances
  Output: local intrinsic dimensionality estimate d̂

  1: d ← distances[distances > 0]         // remove exact-zero entries
  2: if |d| < 2: return 1.0               // degenerate: not enough distinct distances
  3: d_k ← d[-1]                          // distance to the k-th neighbour
  4: log_ratios ← log(d_k / d[:-1])       // log(d_k/d_i) for i = 1..|d|-1
  5: mean_log ← mean(log_ratios)
  6: if mean_log ≤ 0 or not finite: return float(|d|)   // fallback
  7: return 1.0 / mean_log
```

This is the **Levina-Bickel (2004) MLE** for local intrinsic dimensionality. Under a Poisson process with intrinsic dimension $d$, the log-ratios $\log(d_k/d_i)$ are i.i.d. exponential with rate $d$. The MLE of an exponential rate from $m$ i.i.d. samples is $1/\bar{x}$, giving $\hat{d} = 1/\overline{\log(d_k/d_i)}$.

**Fallback on line 6.** `mean_log ≤ 0` indicates all distances are identical — the concentration-of-measure effect in very high dimensions. The fallback `float(|d|)` returns a large value, signalling high effective dimensionality.

**Usage note.** `_estimate_lid` is present in `KNNFairRank` but is **not called by the base decision rule** in v3. It is infrastructure for LID-damped variants that were explored and not retained. It remains because it is used in diagnostic output and is available for future modifications.

### 16.5 `_vote_fraction` — the core algorithm

This method implements the multi-rank binary vote from Section 2.5.

```
PROCEDURE _vote_fraction(x)
  Input:  x ∈ ℝ^d
  Output: (frac_minority, n_votes_cast)

  1:  d_min, d_maj ← _per_class_distances(x)
  2:  if |d_min| = 0 or |d_maj| = 0: return early   // see §16.6

  // Determine effective majority rank (dimension-free fair rank)
  3:  k_eff ← max(1, round(r))

  // Determine how many votes can actually be cast
  4:  max_votes_maj ← |d_maj| // k_eff        // integer floor: how many k_eff-blocks fit
  5:  n ← min(n_votes_param, |d_min|, max_votes_maj)

  // Fallback: guarantee at least one comparison
  6:  if n < 1:
  7:      n ← 1
  8:      k_eff ← min(k_eff, |d_maj|)          // prevent out-of-bounds on line 11

  // Build comparison arrays (fully vectorised)
  9:  min_refs ← d_min[0 : n]                              // [d_1^min, ..., d_n^min]
 10:  maj_indices ← [1·k_eff − 1, 2·k_eff − 1, ..., n·k_eff − 1]   // 0-based indices
 11:  maj_indices ← clip(maj_indices, 0, |d_maj| − 1)     // safety clip
 12:  maj_refs ← d_maj[maj_indices]                        // [d_{k_eff}^maj, ..., d_{n·k_eff}^maj]

  // Cast votes
 13:  votes_minority ← sum(min_refs < maj_refs)            // vectorised boolean sum
 14:  return votes_minority / n, n
```

**Line 3.** `k_eff = round(r)` is the dimension-free fair rank derived in Section 2.4. `max(1, ...)` ensures we never compare against rank 0 (undefined). With $r = 1$ (balanced dataset) `k_eff = 1` — the algorithm degrades to standard 1-NN comparison, which is correct.

**Line 4.** `max_votes_maj = |d_maj| // k_eff` answers: how many complete blocks of size `k_eff` fit in the fetched majority neighbourhood? Vote $i$ accesses majority rank $i \cdot k_\text{eff}$; the maximum $i$ for which this is within bounds is `|d_maj| // k_eff`.

**Line 5.** Three independent caps:
- `n_votes_param`: user-specified maximum
- `|d_min|`: cannot reference a minority rank beyond what was fetched
- `max_votes_maj`: cannot reference a majority rank beyond what was fetched

**Lines 6–8.** If all three caps reduce `n` below 1 (can happen when `k_eff > |d_maj|`), cast exactly one vote. `k_eff` is capped to the number of available majority points to prevent `d_maj[k_eff - 1]` from going out of bounds.

**Line 10.** `maj_indices = [k_eff-1, 2k_eff-1, ..., nk_eff-1]` converts 1-based ranks to 0-based NumPy indices. Vote $i$ pairs $d_i^\text{min}$ with $d_{i \cdot k_\text{eff}}^\text{maj}$; the 0-based array index of the latter is $i \cdot k_\text{eff} - 1$.

**Line 11.** Safety clip after the caps on `n`. After lines 4–5 all indices should already be in bounds, but integer rounding can occasionally produce an off-by-one. The clip is a silent safety net with zero performance cost.

**Line 13.** `min_refs < maj_refs` is a vectorised boolean comparison. `np.sum(...)` counts `True` values (minority wins). The result divided by `n` gives the fraction of votes favouring minority.

### 16.6 Degenerate-class handling

```python
if len(d_min) == 0 or len(d_maj) == 0:
    return (1.0 if len(d_min) > 0 else 0.0, 0)
```

- Only minority points exist → return fraction 1.0 (all votes minority)
- Only majority points exist → return fraction 0.0 (all votes majority)
- The second return value `0` (no votes cast) signals to `_decide` that the comparison was bypassed

These cases occur only in degenerate inner-CV folds where stratification fails on very small datasets.

### 16.7 `_decide` — converting fraction to label

```python
def _decide(self, x):
    if len(self._X_min) == 0:
        return self._majority_class
    if len(self._X_maj) == 0:
        return self._minority_class
    frac_min, n_votes = self._vote_fraction(x)
    if n_votes == 0:
        return self._majority_class
    return self._minority_class if frac_min > 0.5 else self._majority_class
```

**Tie-breaking (`frac_min = 0.5`).** A tie predicts majority. This is **conservative**: it preserves precision at the cost of recall, which is appropriate when the majority class is the "safe" default. With `n_votes = 5` (odd), `frac_min` can only take values 0, 0.2, 0.4, 0.6, 0.8, 1.0, so ties require an even number of votes — possible if the vote count is capped below `n_votes`.

**`n_votes = 0`** falls back to majority. This is the degenerate case where `_vote_fraction` returned early.

### 16.8 `_predict_x` and `_predict_proba_x`

```python
def _predict_x(self, x):
    return self._decide(x)

def _predict_proba_x(self, x):
    prob = np.zeros(len(self.classes_))
    ...
    frac_min, n_votes = self._vote_fraction(x)
    prob[idx_min] = frac_min
    prob[idx_maj] = 1.0 - frac_min
    return prob
```

Both delegate to `_vote_fraction`. The probability vector is indexed by `self.classes_` — not hardcoded to minority=index 0 — so it is correct regardless of label encoding (e.g. minority class labelled 1 or labelled -1).

### 16.9 Complexity summary

| Phase | Cost |
|---|---|
| `fit` | $O(N)$ — counting, splitting, two integers |
| `_per_class_distances` per query | $O(N_\text{min} \cdot d + N_\text{maj} \cdot d)$ for `cdist`; $O(k_\text{min\_eff} \log k_\text{min\_eff} + k_\text{maj\_eff} \log k_\text{maj\_eff})$ for sort |
| `_vote_fraction` per query (after distances) | $O(n_\text{votes})$ |
| **Total per query** | $O(N \cdot d)$ |
| **Test set of $M$ queries** | $O(M \cdot N \cdot d)$ |

This is asymptotically identical to standard KNN. The rank correction adds no extra asymptotic cost — it only changes which neighbours are compared.

<a id="knnfairrankb"></a>

---
## 10. KNNFairRankMagnitude — Modification B Implementation

`KNNFairRankMagnitude` is defined in `src/algorithms/fair_rank/core/knn_fair_rank_b.py`. It is the thinnest possible subclass of `KNNFairRank`: it overrides only `_vote_fraction`, replacing the binary comparison with a continuous confidence score.

### 17.1 What changes

In `KNNFairRank._vote_fraction` each comparison yields a binary outcome:

```python
votes_minority = int(np.sum(min_refs < maj_refs))
return votes_minority / n_votes, n_votes
```

In `KNNFairRankMagnitude._vote_fraction` each comparison yields a continuous score:

```python
denom = min_refs + maj_refs
safe = denom > 0
scores = np.where(safe, maj_refs / np.where(safe, denom, 1.0), 0.5)
return float(np.mean(scores)), n_votes
```

Everything else — neighbourhood sizing, `k_eff` computation, vote count caps, fallback logic — is inherited unchanged from `KNNFairRank`.

### 17.2 The continuous score formula

For comparison $i$, the score is:

$$s_i = \frac{d_{i \cdot k_\text{eff}}^\text{maj}}{d_i^\text{min} + d_{i \cdot k_\text{eff}}^\text{maj}} \in [0, 1]$$

Intuition:
- $s_i = 0.5$ → both distances are equal (exactly on the v3 decision boundary — neutral evidence)
- $s_i \to 1$ → majority reference is much further than minority (strong minority evidence)
- $s_i \to 0$ → minority reference is much further than majority (strong majority evidence)

The hard decision rule is unchanged: predict minority iff $\bar{s} = \frac{1}{n_\text{votes}}\sum_i s_i > 0.5$.

At $\bar{s} = 0.5$ the tie-breaking is the same as v3 (predicts majority). The boundary between the two classes is still the set of points where $\bar{s} = 0.5$, but unlike binary voting, points far inside a class region have $\bar{s}$ close to 0 or 1, and near-boundary points have $\bar{s}$ close to 0.5. This is meaningful for `predict_proba`.

### 17.3 The numerical guard

The guard `safe = denom > 0` handles the edge case where both distances are exactly zero — the query point coincides with a training point of both classes simultaneously. This is geometrically impossible in general position but can occur after standardisation on datasets with duplicate rows carrying different labels.

The expression `np.where(safe, denom, 1.0)` in the denominator avoids a division-by-zero warning. NumPy evaluates both branches of `np.where` before selecting, so `maj_refs / denom` would raise a warning wherever `denom = 0` even if those positions are masked out in the final result. Substituting `1.0` as a dummy denominator in the unsafe positions suppresses the warning. The value `0.5` is selected for those positions in the outer `np.where` and the dummy denominator is never used in the output.

### 17.4 What it improves and what it does not

**Improves:** ROC AUC and probability calibration. The `predict_proba` output is a smooth continuous function of the distance ratio rather than a quantised fraction $i/n_\text{votes}$ (which can only take values 0, 0.2, 0.4, 0.6, 0.8, 1.0 with 5 votes). Downstream threshold-tuning or cost-sensitive decision-making benefits from better-calibrated soft scores.

**Does not improve:** G-mean or balanced accuracy. Those metrics depend on the hard decision at the default threshold, and the 0.5 threshold on $\bar{s}$ produces the same decision boundary as the binary vote fraction. The empirical improvement in ROC AUC (~+1.6 points, Holm-corrected Wilcoxon $p \approx 2 \times 10^{-4}$) does not translate into a detectable change in hard-label metrics.

<a id="knnfairrankc"></a>

---
## 11. KNNFairRankCV — Modification C Implementation

`KNNFairRankCV` is defined in `src/algorithms/fair_rank/core/knn_fair_rank_c.py`. It extends `KNNFairRank` by introducing a single exponent α that scales the rank correction to $k_\text{eff} = r^\alpha$, and selects α by inner stratified cross-validation.

### 18.1 Constructor

```python
KNNFairRankCV(
    alpha_grid=[0.25, 0.5, 0.75, 1.0],
    inner_cv_folds=3,
    scoring="geometric_mean",
    # plus all KNNFairRank parameters (k_min, k_maj_buffer, ...)
)
```

`alpha_grid` is the set of candidate exponents. The grid is deliberately biased toward α = 1 (the theoretically correct value): α = 0 (no correction) is excluded because a dataset that genuinely cannot benefit from correction is already handled by `KNNOptK`.

All `KNNFairRank` parameters pass through to `super().__init__` unchanged. Crucially, the majority neighbourhood `k_maj` is always sized for α = 1 (the worst case — largest `k_eff`). For any α < 1, `k_eff` is smaller, so fewer majority neighbours are needed and the pre-sized neighbourhood always has enough. **No refitting is required when evaluating different α values.**

### 18.2 `_score_alpha` — the α-evaluation trick

```
PROCEDURE _score_alpha(X, y, α)
  Input:  training data, a candidate exponent
  Output: mean inner-CV score using k_eff = r^α

  1: cv ← StratifiedKFold(inner_cv_folds, shuffle=True, seed)
  2: for each (train_idx, val_idx) in cv.split(X, y):
  3:     clf ← KNNFairRank(k_min, k_maj_buffer, ...)   // all other params from self
  4:     clf.fit(X[train_idx], y[train_idx])            // fits with r = N_maj/N_min
  5:     clf._r ← clf._r ** α                          // reinterpret: r becomes r^α
  6:     y_pred ← clf.predict(X[val_idx])               // _vote_fraction uses modified _r
  7:     scores.append(score_fn(y[val_idx], y_pred))
  8: return mean(scores)
```

**Why the trick on line 5 works.** After `clf.fit`, `clf._r = N_maj/N_min`. Setting `clf._r = (N_maj/N_min)**α` makes subsequent calls to `_vote_fraction` compute `k_eff = round(self._r)` which evaluates to `round(r^α)` — without any other state change. The neighbourhood sizes `_k_maj_eff` and `_k_min_eff` were set at fit time for α = 1 and remain valid because α ≤ 1 implies a smaller `k_eff` and lower demand on the majority neighbourhood.

This means: fit once per fold, evaluate all α values by mutating `_r` in sequence. The alternative — fitting a new `KNNFairRank` for every (fold, α) pair — would multiply cost by `|alpha_grid|`.

### 18.3 `fit` — outer α-selection loop

```
PROCEDURE KNNFairRankCV.fit(X, y)
  Input:  X ∈ ℝ^{N×d}, y ∈ {0,1}^N
  Output: fitted model; self._alpha and self.best_alpha_ set

  1: best_alpha ← 1.0
  2: best_score ← −∞
  3: for α in alpha_grid:
  4:     score ← _score_alpha(X, y, α)
  5:     if score > best_score:
  6:         best_score ← score
  7:         best_alpha ← α
  8: self._alpha ← best_alpha
  9: self.best_alpha_ ← best_alpha       // public alias for inspection
 10: super().fit(X, y)                   // fit outer model on all training data
```

After selecting α, the outer model is fit by calling `KNNFairRank.fit` on the full training set. From this point `self._r` holds the raw ratio $N_\text{maj}/N_\text{min}$. The α scaling is applied inside `_vote_fraction` at prediction time.

### 18.4 `_vote_fraction` override

The only change from `KNNFairRank._vote_fraction` is line 3:

```
v3:   k_eff ← max(1, round(self._r))
CV:   k_eff ← max(1, round(self._r ** self._alpha))
```

Everything else — the vote count caps, vectorised comparison, fallback — is identical. When `self._alpha = 1.0` this is bit-for-bit equal to v3.

### 18.5 `best_alpha_` as a diagnostic

After fitting, `clf.best_alpha_` is publicly accessible. Its distribution across datasets in a benchmark is diagnostic:
- α = 1.0 selected most often → Poisson-uniform assumption holds for most datasets
- α < 1.0 selected often → v3 over-corrects; the assumption is frequently violated
- α = 0.25 selected often → minority class is heavily clustered or severely imbalanced with few samples

### 18.6 Complexity

Inner CV adds `|alpha_grid| × inner_cv_folds` fits of `KNNFairRank`, each on $\approx (1 - 1/\text{folds})N$ points. With defaults (4 alphas, 3 folds), this is 12 inner fits. Prediction complexity per query is identical to `KNNFairRank`.

<a id="knnfairrankbc"></a>

---
## 12. KNNFairRankMagnitudeCV — Combined Modification B + C

**Addresses:** F3 (magnitude) and F4 (correction strength)

**Source:** `src/algorithms/fair_rank/core/knn_fair_rank_bc.py`

**Extends:** `KNNFairRankCV`

### 12.1 Motivation

`KNNFairRankMagnitude` (Mod B) wins ROC AUC in isolation; `KNNFairRankCV` (Mod C) wins
G-mean. The two modifications are orthogonal: B changes *how votes are aggregated*
(binary → soft score); C changes *which majority rank is consulted* (global $r$ →
$r^\alpha$ with data-driven $\alpha$). Combining them is therefore natural.

The consistency requirement is important: if $\alpha$ is selected by inner CV using
binary votes but deployed with magnitude-aware votes, the selection criterion is
misaligned with the deployment criterion. `KNNFairRankMagnitudeCV` avoids this by
using `KNNFairRankMagnitude` instances inside the inner CV loop.

### 12.2 Implementation

`KNNFairRankMagnitudeCV` inherits from `KNNFairRankCV` and overrides only
`_score_alpha`. The parent's `fit` logic (CV loop, best $\alpha$ selection, final
refitting at $\alpha_\text{best}$) is unchanged. The override substitutes
`KNNFairRankMagnitude` for `KNNFairRank` in the inner CV fold training:

```python
class KNNFairRankMagnitudeCV(KNNFairRankCV):
    def _score_alpha(self, X, y, alpha):
        for tr, va in cv.split(X, y):
            clf = KNNFairRankMagnitude(k_min=..., k_maj_buffer=..., k_maj_floor=...)
            clf._r = original_r ** alpha   # rescale the rank correction
            clf.fit(X[tr], y[tr])
            scores.append(score_fn(y[va], clf.predict(X[va])))
        return np.mean(scores)
```

After CV selects $\alpha_\text{best}$, the parent's `fit` refits a fresh
`KNNFairRankMagnitude` at that $\alpha$ (not a `KNNFairRank`), so the deployed
model uses magnitude-aware voting at the CV-optimal correction strength.

### 12.3 Constructor

```python
KNNFairRankMagnitudeCV(
    alpha_grid      = [0.25, 0.5, 0.75, 1.0],
    inner_cv_folds  = 3,
    scoring         = "geometric_mean",
    # plus all KNNFairRank parameters
)
```

Identical to `KNNFairRankCV`. The magnitude-awareness is transparent to the caller.

<a id="knnfairrankens"></a>

---
## 13. KNNFairRankEnsemble — α-Grid Ensemble

**Addresses:** F4 (correction strength), variance reduction alternative to CV

**Source:** `src/algorithms/fair_rank/ensemble/knn_fair_rank_ens.py`

**Extends:** `KNNFairRank`

### 13.1 Motivation

`KNNFairRankCV` selects one $\alpha$ per dataset by inner CV. On small datasets
(few minority points) the CV estimate is noisy: different splits may select different
$\alpha$ values. An ensemble over the entire $\alpha$-grid avoids this commitment at
the cost of never being exactly optimal for any individual dataset.

### 13.2 Voting scheme

At prediction time, a soft vote fraction is computed for every $(n_\text{votes}, \alpha)$
pair in the grid:

$$\hat{p}(x) = \frac{1}{|\mathcal{V}| \cdot |\mathcal{A}|}
\sum_{v \in \mathcal{V}} \sum_{\alpha \in \mathcal{A}}
\frac{1}{v}\sum_{i=1}^{v} \mathbf{1}\!\left[d_i^\text{min} < d_{\lfloor i \cdot r^\alpha \rceil}^\text{maj}\right]$$

All comparisons read from the **same pre-sorted majority distance array**, sized for
$k_\text{max} = \max_{v,\alpha}\{\lfloor v \cdot r^\alpha \rceil\}$. No re-fitting or
inner CV is needed at training time.

### 13.3 Implementation

`KNNFairRankEnsemble` overrides `predict` and `predict_proba`. The `fit` is inherited
from `KNNFairRank` unchanged. The sole constructor addition is `alpha_grid`:

```python
KNNFairRankEnsemble(
    alpha_grid = [0.25, 0.5, 0.75, 1.0],   # None → reads from settings.yaml
    # plus all KNNFairRank parameters
)
```

Internally, `fit` is called with an enlarged majority neighbourhood
($k_\text{maj}$ sized for the largest $k_\text{eff}$ in the grid) so that every
$\alpha$ value can access the majority ranks it needs.

<a id="knnfairrankbens"></a>

---
## 14. KNNFairRankMagnitudeEnsemble — Magnitude Voting + α-Grid Ensemble

**Addresses:** F3 + F4

**Source:** `src/algorithms/fair_rank/ensemble/knn_fair_rank_b_ens.py`

**Extends:** `KNNFairRankMagnitude`

### 14.1 Combination

`KNNFairRankMagnitudeEnsemble` combines the continuous score from Modification B
with the $\alpha$-grid ensemble from `KNNFairRankEnsemble`, replacing the inner CV
of `KNNFairRankMagnitudeCV`.

For each $(v, \alpha, i)$ triple, the soft score is:

$$s_{v,\alpha,i}(x) = \frac{d_{\lfloor i \cdot r^\alpha \rceil}^\text{maj}(x)}
{d_i^\text{min}(x) + d_{\lfloor i \cdot r^\alpha \rceil}^\text{maj}(x)} \in (0,1)$$

The final prediction is minority iff the mean over all triples exceeds 0.5.

### 14.2 Constructor

```python
KNNFairRankMagnitudeEnsemble(
    alpha_grid = [0.25, 0.5, 0.75, 1.0],
    # plus all KNNFairRankMagnitude parameters
)
```

Inherits from `KNNFairRankMagnitude`; overrides `_vote_fraction` to sweep the
$\alpha$-grid and return the mean continuous score.

<a id="knnfairrankoptvotes"></a>

---
## 15. KNNFairRankOptVotes — CV-Tuned Vote Count

**Addresses:** F4 (correction strength), bimodal $n_\text{votes}$ distribution

**Source:** `src/algorithms/fair_rank/ensemble/knn_fair_rank_opt_votes.py`

**Extends:** `KNNFairRankCV`

### 15.1 Motivation

The empirical $n_\text{votes}$ sweep (Section 3, code cell) reveals a bimodal distribution
of per-dataset optima: many datasets peak at $n_\text{votes} = 1$–$2$ (a tight single
comparison is sufficient), while others peak at $n_\text{votes} \geq 10$ (averaging
over many ranks is necessary). A fixed default of $n_\text{votes} = 5$ is never
catastrophically wrong but leaves room for improvement on both ends.

### 15.2 Implementation

`KNNFairRankOptVotes` adds an outer loop over `n_votes_grid` to `KNNFairRankCV`'s
inner $\alpha$ loop. Both hyperparameters are tuned by the same inner stratified CV,
but the search is **greedy** rather than joint: for each candidate $n_\text{votes}$,
the best $\alpha$ is found independently, and the $(n_\text{votes}, \alpha)$ pair
with the highest inner CV score is selected.

### 15.3 Constructor

```python
KNNFairRankOptVotes(
    n_votes_grid    = [1, 2, 3, 5, 7, 10, 15],   # None → settings.yaml
    alpha_grid      = [0.25, 0.5, 0.75, 1.0],
    inner_cv_folds  = 3,
    scoring         = "geometric_mean",
    # plus all KNNFairRank parameters
)
```

After fitting, `best_n_votes_` and `best_alpha_` are set as public attributes.

<a id="knnfairrankjointcv"></a>

---
## 16. KNNFairRankJointCV — Joint CV over $n_\text{votes}$ and $\alpha$

**Addresses:** F4, bimodal $n_\text{votes}$ distribution

**Source:** `src/algorithms/fair_rank/ensemble/knn_fair_rank_joint_cv.py`

**Extends:** `KNNFairRank`

### 16.1 Motivation and efficiency insight

`KNNFairRankOptVotes` tunes $n_\text{votes}$ and $\alpha$ greedily (one dimension at a
time). The true joint optimum may not be on the greedy path. `KNNFairRankJointCV`
searches the full joint grid, but does so efficiently.

The key observation: a model trained at $n_\text{votes} = v$ can evaluate any $\alpha$
value at no extra cost, because different $\alpha$ values only differ in which rank of
the pre-sorted majority distance array is read. The joint grid has
$|\mathcal{V}| \times |\mathcal{A}|$ combinations, but the inner CV only needs
$|\mathcal{V}|$ training fits per fold (one per $n_\text{votes}$ value).

### 16.2 Inner CV loop structure

```
for each CV fold (tr, va):
    for each v in n_votes_grid:          # |V| fits per fold
        clf_v.fit(X[tr], y[tr])
        for each α in alpha_grid:        # only predict calls — free
            score[v, α] = scorer(y[va], clf_v.predict_at_alpha(X[va], α))

best_v, best_α = argmax score[v, α]
final_clf.fit(X, y) at best_v, best_α
```

### 16.3 Constructor

```python
KNNFairRankJointCV(
    n_votes_grid    = [1, 2, 3, 5, 7, 10, 15],
    alpha_grid      = [0.25, 0.5, 0.75, 1.0],
    inner_cv_folds  = 3,
    scoring         = "geometric_mean",
    # plus all KNNFairRank parameters
)
```

Public attributes after fitting: `best_n_votes_`, `best_alpha_`.

<a id="knnfairranklocal-odds"></a>

---
## 17. KNNFairRankLocalOdds — Per-Query Local Odds (Rank-Pair Interpolation)

**Addresses:** F1, F2 (local density heterogeneity)

**Source:** `src/algorithms/fair_rank/local/knn_fair_rank_local_odds.py`

**Extends:** `KNNFairRank`

### 17.1 Idea

Replace the global $k_\text{eff} = r$ with a **per-query** estimate derived from the
class-separated distance arrays already computed by the base class. For each minority
rank $i = 1, \ldots, k_\text{probe}$, count how many majority training points are
closer to $x$ than the $i$-th minority neighbour:

$$j_i(x) = \#\{x_j^\text{maj} : \|x_j^\text{maj} - x\| < d_i^\text{min}(x)\}$$

This is a single `searchsorted` call on the sorted majority distances. Under a
homogeneous Poisson process, $E[j_i / i] = r$. Under locally dense minority
($d_i^\text{min}$ small), $j_i / i < r$; under locally dense majority, $j_i / i > r$.

The $k_\text{probe}$ estimates are combined as a median (robust to individual outlier
minority distances) and Bayesian-shrunk toward the global $r$ to stabilise sparse
regions:

$$k_\text{eff}(x) = \text{round}\!\left(
    \frac{n_\text{shrink} \cdot r + k_\text{probe} \cdot \hat{r}_\text{local}(x)}
         {n_\text{shrink} + k_\text{probe}}
\right)$$

where $n_\text{shrink}$ is a prior strength hyperparameter (default 5).

### 17.2 Why this avoids the d-estimation problem

The ratio $j_i / i$ is a pure counting operation — no intrinsic dimensionality $d$
appears. It avoids the $(d_\text{min}/d_\text{maj})^d$ amplification that made the
direct density-ratio formulation fragile (see Section 4, failure mode F1).

### 17.3 Constructor

```python
KNNFairRankLocalOdds(
    k_probe      = None,    # minority ranks used to estimate k_eff; None → adaptive
    n_shrink     = 5,       # Bayesian prior strength toward global r
    # plus all KNNFairRank parameters
)
```

Overrides `_vote_fraction` to compute per-query $k_\text{eff}$ from the sorted distance
arrays before running the standard binary vote.

<a id="knnfairranklocalcount"></a>

---
## 18. KNNFairRankLocalCount — Per-Query Minority-Budget Counting

**Addresses:** F1, F2

**Source:** `src/algorithms/fair_rank/local/knn_fair_rank_local_count.py`

**Extends:** `KNNFairRank`

### 18.1 Idea

Fix the **minority denominator** at exactly $k_\text{ref}$ (no Poisson noise), then
count how many majority points fall within the same radius:

$$\text{radius}(x) = d_{k_\text{ref}}^\text{min}(x), \qquad
k_\text{eff}(x) = \max\!\left(1,\; \text{round}\!\left(
    \frac{\#\{x_j^\text{maj} : \|x_j^\text{maj} - x\| \leq \text{radius}(x)\}}{k_\text{ref}}
\right)\right)$$

This requires one binary search on the sorted majority distance array per query
(no new $k$-NN passes). The estimate reduces to the global $r$ under uniform density
and adapts in both directions otherwise.

### 18.2 Default $k_\text{ref}$

The adaptive default is $k_\text{ref} = \max(3, \lfloor\sqrt{N_\text{min}}\rfloor)$.
This gives a minority denominator that grows slowly with dataset size, balancing
estimate reliability against locality.

### 18.3 Constructor

```python
KNNFairRankLocalCount(
    k_ref = None,   # None → adaptive default at fit time
    # plus all KNNFairRank parameters
)
```

Overrides `_vote_fraction`. The implementation is almost identical to
`KNNFairRankLocalOdds` but uses a fixed minority denominator instead of a median
over rank-pair ratios.

<a id="knnfairrankbayesian"></a>

---
## 19. KNNFairRankBayesian — Bayesian α-Shrinkage

**Addresses:** F1, F2 (locally), F4 (globally via prior)

**Source:** `src/algorithms/fair_rank/local/knn_fair_rank_bayesian.py`

**Extends:** `KNNFairRankJointCV`

### 19.1 Idea

`KNNFairRankJointCV` selects a global correction exponent $\alpha_\text{CV}$ that is
optimal on average across the dataset. This is a high-bias estimator: it applies the
same $k_\text{eff} = r^{\alpha_\text{CV}}$ everywhere.

`KNNFairRankBayesian` refines that global estimate per query using the
per-class counting signal from Section 18, blended via a **Bühlmann credibility weight**:

1. **Local density signal** (§20 per-class counting):
   $\hat{\rho}(x) = \#\{x_j^\text{maj} : \|x_j^\text{maj} - x\| \leq d_{k_\text{ref}}^\text{min}(x)\} \,/\, k_\text{ref}$

2. **Map to $\alpha$-space:**
   $\alpha_\text{local}(x) = \log(\hat{\rho}(x)) / \log(r)$

3. **Bühlmann credibility weight** — prior strength proportional to $k_\text{ref}$,
   evidence strength proportional to local majority count $n_\text{maj}^\text{local}$:

$$w(x) = \frac{k_\text{ref}}{k_\text{ref} + n_\text{maj}^\text{local}(x)}$$

   When $n_\text{maj}^\text{local} \approx 0$ (minority-dense region, sparse majority
   evidence): $w \to 1$ — fall back entirely to the global CV prior.
   When $n_\text{maj}^\text{local} \gg k_\text{ref}$ (majority-dense region): $w \to 0$
   — trust the local signal.

4. **Blended exponent and final $k_\text{eff}$:**

$$\alpha_\text{final}(x) = w(x)\,\alpha_\text{CV} + (1-w(x))\,\alpha_\text{local}(x), \qquad
k_\text{eff}(x) = \text{round}(r^{\alpha_\text{final}(x)})$$

### 19.2 Constructor

```python
KNNFairRankBayesian(
    k_ref = None,   # counting reference — None → adaptive default
    # plus all KNNFairRankJointCV parameters
)
```

The joint CV runs at training time to establish $\alpha_\text{CV}$; the Bayesian
blending runs at prediction time per query.

<a id="knnfairrankdensityregion"></a>

---
## 20. KNNFairRankDensityRegion — Dual-Class Density Regions

**Addresses:** F1, F2

**Source:** `src/algorithms/fair_rank/local/knn_fair_rank_density_region.py`

**Extends:** `KNNFairRank`

### 20.1 Architecture

`KNNFairRankDensityRegion` estimates local density by partitioning the feature space
into regions using separate cluster representations for each class, then assigning each
query to its nearest region and applying the corresponding local $k_\text{eff}$.

**Fit:**

1. For every minority training point $x_i^\text{min}$, estimate the local density
   ratio using the per-class counting formula (Section 18):
   $\hat{\rho}(x_i^\text{min}) = n_\text{maj}^\text{local} / k_\text{ref}$.

2. Sort the $\hat{\rho}$ values and gap-detect a partition into $K$ density levels,
   each containing at least $k_\text{ref}$ minority points (reliability guard).

3. Compute one minority centroid per region:
   $C_k^\text{min} = \text{mean}(X_\text{min} \text{ in region } k)$.

4. Run $K$-means on $X_\text{maj}$ to obtain $K$ majority centroids $C_j^\text{maj}$
   with corresponding cluster sizes $n_j^\text{maj}$.

5. Store $k_\text{eff}$ per $(k, j)$ region pair:
   $k_\text{eff}(k, j) = n_j^\text{maj} / k$.

**Predict:**

For each query $x$: find nearest minority centroid $k^*$ and nearest majority centroid
$j^*$; use $k_\text{eff}(k^*, j^*)$ as the majority rank for voting.

### 20.2 Constructor

```python
KNNFairRankDensityRegion(
    k_ref      = None,   # counting reference; None → adaptive
    n_regions  = None,   # K; None → gap-detected automatically
    # plus all KNNFairRank parameters
)
```

<a id="knnfairrankjackknife"></a>

---
## 21. KNNFairRankJackknife — LOO Jackknife over Minority Ranks

**Addresses:** F3 (outlier minority points distort vote fractions)

**Source:** `src/algorithms/fair_rank/resampling/knn_fair_rank_jackknife.py`

**Extends:** `KNNFairRank`

### 21.1 The outlier minority problem

The vote fraction in `KNNFairRank._vote_fraction` is computed from the $n_\text{votes}$
closest minority neighbours. A single outlier minority point — a mislabelled majority
sample or a genuine minority point far from its cluster — sits deep in majority territory.
Its $d_j^\text{min}$ is large relative to the corrected majority distance, so it
consistently votes majority, biasing the fraction downward for all queries in its vicinity.

### 21.2 LOO correction

For each query $x$, compute the vote fraction $n_\text{votes}$ times, each time removing
one minority neighbour and re-running the comparisons on the shifted rank sequence:

$$\hat{p}_{\backslash j}(x) = \frac{1}{n_\text{votes} - 1}
\sum_{\substack{i=1\\i \neq j}}^{n_\text{votes}}
\mathbf{1}\!\left[d_i^\text{min} < d_{\lfloor i \cdot r \rceil}^\text{maj}\right]$$

The final vote fraction is the average of the $n_\text{votes}$ leave-one-out estimates:

$$\hat{p}^\text{JK}(x) = \frac{1}{n_\text{votes}} \sum_{j=1}^{n_\text{votes}} \hat{p}_{\backslash j}(x)$$

An outlier at position $j^*$ contributes its biasing vote to only
$(n_\text{votes} - 1)$ of the LOO estimates and is excluded from one. Its influence
is diluted by a factor of approximately $n_\text{votes}$.

### 21.3 Cost

The LOO loop requires $n_\text{votes}$ passes over an already-fetched distance array.
No new $k$-NN lookups are needed. The computational overhead over `KNNFairRank` is
$O(n_\text{votes}^2)$ arithmetic operations per query — negligible in practice.

### 21.4 Constructor

Identical to `KNNFairRank`. The LOO mechanism is activated via the overridden
`_vote_fraction` method with no new hyperparameters.

<a id="knnfairranklid"></a>

---
## 22. KNNFairRankLID — LID-Derived α (No Inner CV)

**Addresses:** F4

**Source:** `src/algorithms/fair_rank/local/knn_fair_rank_lid.py`

**Extends:** `KNNFairRankCV`

### 22.1 Idea

`KNNFairRankCV` selects $\alpha$ by inner cross-validation — an empirical approach.
`KNNFairRankLID` derives $\alpha$ analytically from the **coefficient of variation of
local intrinsic dimensionality (LID)** estimates across the training set, with no inner
CV loop.

### 22.2 Why CV of LID (not mean LID)

Mean LID measures intrinsic dimensionality. Since $d$ cancels in the FairRank
derivation, mean LID is not the right predictor of $\alpha$.

The coefficient of variation $\text{CV}_\text{LID} = \sigma_\text{LID} / \mu_\text{LID}$
measures **how non-uniform the local geometry is** across the dataset. High CV means
density varies strongly from region to region — exactly the condition under which the
global $r$ over-corrects and $\alpha < 1$ is warranted.

### 22.3 Formula

$$\alpha = \text{clip}\!\left(1 - s \cdot \text{CV}_\text{LID},\; \alpha_\text{min},\; 1.0\right)$$

where $s$ is a sensitivity scale (default 0.75). Properties:

- $\text{CV}_\text{LID} = 0$ (perfectly uniform) $\Rightarrow$ $\alpha = 1.0$
- $\text{CV}_\text{LID} = 1/s$ $\Rightarrow$ $\alpha = 0$ (clamped to $\alpha_\text{min}$)
- Intermediate values interpolate smoothly

LID is estimated at every training point using the Levina–Bickel MLE over the $k_\text{lid}$
nearest neighbours. The entire $\alpha$ derivation runs in `fit` with no CV folds.

### 22.4 Constructor

```python
KNNFairRankLID(
    k_lid       = 20,    # neighbourhood size for LID estimation
    scale       = 0.75,  # sensitivity of alpha to CV_LID
    alpha_min   = 0.25,  # lower clamp
    # plus all KNNFairRank parameters
)
```

Sets `best_alpha_` at fit time (same interface as `KNNFairRankCV`).

<a id="knnfairrankjackknifeens"></a>

---
## 23. KNNFairRankJackknifeEnsemble — LOO Jackknife + α-Grid Ensemble

**Addresses:** F3 + F4

**Source:** `src/algorithms/fair_rank/resampling/knn_fair_rank_jackknife_ens.py`

**Extends:** `KNNFairRank`

### 23.1 Combination

`KNNFairRankJackknifeEnsemble` stacks both robustness mechanisms from Sections 21
and 13: for each LOO trial $j$ (remove $d_j^\text{min}$, shift ranks), the full
$\alpha$-grid vote is run on the shifted minority sequence. All $(j, \alpha, i)$ outcomes
are averaged into a single fraction.

This tests whether the two sources of improvement are additive:
- The jackknife reduces sensitivity to individual outlier minority distances.
- The $\alpha$-grid introduces the conservative correction ($\alpha < 1$ values) that
  improved precision and MCC in `KNNFairRankEnsemble`.

### 23.2 Fallback

If $k_\text{min}$ is too small to run any LOO trial
(fewer than $n_\text{votes} + 1$ minority distances), the method falls back to the
pure $\alpha$-grid ensemble vote, identical to `KNNFairRankEnsemble`.

### 23.3 Constructor

```python
KNNFairRankJackknifeEnsemble(
    alpha_grid = [0.25, 0.5, 0.75, 1.0],
    # plus all KNNFairRank parameters
)
```

<a id="knnfairranklocaloddsjk"></a>

---
## 24. KNNFairRankLocalOddsJackknife — Local Odds + LOO Jackknife

**Addresses:** F1, F2, F3

**Source:** `src/algorithms/fair_rank/resampling/knn_fair_rank_local_odds_jackknife.py`

**Extends:** `KNNFairRankLocalOdds`

### 24.1 Motivation

In `KNNFairRankLocalOdds`, the per-query $k_\text{eff}$ is estimated from the ratios
$j_i / i$ for $i = 1, \ldots, k_\text{probe}$. A single anomalously close minority
point at position $j^*$ pushes all ratios $j_i / i$ for $i \leq j^*$ downward,
potentially underestimating $k_\text{eff}$ and weakening the correction.

`KNNFairRankLocalOddsJackknife` applies LOO to the $k_\text{eff}$ estimation step:
for each LOO trial $j$, remove $d_j^\text{min}$ from the minority sequence, re-estimate
$k_\text{eff}$ from the shifted ratios, and cast the binary vote with that estimate.
The final vote fraction is the average over all LOO trials.

### 24.2 Constructor

```python
KNNFairRankLocalOddsJackknife(
    k_probe  = None,   # controls both estimation depth and LOO trial count
    n_shrink = 5,
    # plus all KNNFairRank parameters
)
```

Inherits everything from `KNNFairRankLocalOdds`; overrides `_vote_fraction` to add
the LOO outer loop around the odds estimation and voting step.

<a id="knnfairranktopojoint"></a>

---
## 25. KNNFairRankTopoJoint — Topology-Guided Regional Correction

**Addresses:** F1, F2

**Source:** `src/algorithms/fair_rank/topology/knn_fair_rank_topo_joint.py`

**Extends:** `KNNFairRank`

### 25.1 The joint-cloud approach

Per-query local density estimators (Sections 17–20) all suffer from the same problem:
reliable statistics from the minority class in a local window are unavailable under
severe imbalance — exactly when a correction is most needed.

`KNNFairRankTopoJoint` escapes by estimating density from the **joint point cloud**
(all training points regardless of class). Cluster structure is estimated from the
combined cloud (where the majority's larger sample size dominates); minority counts
are only needed at the region level, where aggregation reduces variance.

### 25.2 Algorithm

**Fit:**

1. Run Ward hierarchical clustering on the joint training set $X_\text{train}$.
2. Cut the dendrogram at a depth such that each leaf region contains at least
   $\tau = \max(10, \lfloor\sqrt{n}\rfloor)$ points.
3. For each region $R_i$, compute the local correction:

$$k_\text{eff}(R_i) = \min\!\left(r,\;
    \frac{N_\text{maj}(R_i) + \lambda}{N_\text{min}(R_i) + \lambda}
\right)$$

   where $\lambda = 1$ is Laplace smoothing. Regions where
   $N_\text{min}(R_i) < \tau_\text{min}$ inherit the global $r^\alpha$.

**Predict:**

Assign each query $x$ to its nearest region centroid (Euclidean distance); use that
region's $k_\text{eff}$ as the majority rank for the standard binary vote.

### 25.3 Constructor

```python
KNNFairRankTopoJoint(
    min_region_samples = None,   # τ; None → adaptive max(10, floor(sqrt(n)))
    laplace            = 1,
    # plus all KNNFairRank parameters
)
```

<a id="knnfairranktopojointbs"></a>

---
## 26. KNNFairRankTopoJointBootstrap — Bootstrap-Gated Topology

**Addresses:** F1, F2 (with reliability gating)

**Source:** `src/algorithms/fair_rank/topology/knn_fair_rank_topo_joint_bootstrap.py`

**Extends:** `KNNFairRankTopoJoint`

### 26.1 The unreliability problem

`KNNFairRankTopoJoint` always uses the local $k_\text{eff}(R_i)$ even when
$N_\text{min}(R_i)$ is small and the estimate is noisy. For regions with few minority
points the local correction may be worse than the global one.

### 26.2 Bootstrap reliability gate

`KNNFairRankTopoJointBootstrap` measures per-region reliability via out-of-bag (OOB)
bootstrap evaluation:

1. Draw $B$ bootstrap resamples of the training set.
2. For each resample $b$: fit the Ward partition and generate OOB predictions.
3. Per region $R_i$: compare OOB G-mean using local $k_\text{eff}(R_i)$ against OOB
   G-mean using the global `KNNFairRankJointCV` correction.
4. Mark region $R_i$ as **reliable** if its local correction outperforms the global
   baseline by margin $\delta$ across bootstrap resamples.

Formally:

$$\text{reliable}(R_i) = \mathbf{1}\!\left[
    \frac{1}{B}\sum_{b=1}^{B} G_b^\text{OOB}(R_i, k_\text{eff}^\text{local}) >
    \frac{1}{B}\sum_{b=1}^{B} G_b^\text{OOB}(R_i, k_\text{eff}^\text{JointCV}) + \delta
\right]$$

**Predict:** reliable regions use local $k_\text{eff}$; unreliable regions fall back to
`KNNFairRankJointCV`. The fallback floor is therefore the best global algorithm.

### 26.3 Constructor

```python
KNNFairRankTopoJointBootstrap(
    n_bootstrap        = 20,
    reliability_margin = 0.01,   # δ
    min_region_samples = None,
    laplace            = 1,
    # plus all KNNFairRankTopoJoint / KNNFairRankJointCV parameters
)
```

<a id="knnfairranktopocount"></a>

---
## 27. KNNFairRankTopoCount — Topology-Derived Scale for Counting

**Addresses:** F1, F2 (topology-guided scale selection)

**Source:** `src/algorithms/fair_rank/topology/knn_fair_rank_topo_count.py`

**Extends:** `KNNFairRank`

**Requires:** `ripser` (falls back gracefully if not installed)

### 27.1 The arbitrary scale problem

`KNNFairRankLocalCount` (Section 18) anchors the counting radius to
$d_{k_\text{ref}}^\text{min}(x)$ — the distance to the $k_\text{ref}$-th minority
neighbour. The choice of $k_\text{ref}$ is arbitrary: too small and the ball is noisy;
too large and it blends distinct density regions.

`KNNFairRankTopoCount` replaces this arbitrary scale with the **topology-derived scale**
$\varepsilon^*(x)$: the largest radius at which $x$'s local neighbourhood is still a
single coherent connected component, found via the Vietoris–Rips filtration.

### 27.2 Topology-derived scale

For each query $x$, extract the $k_\text{max}$ nearest training neighbours (joint,
both classes). Build the Vietoris–Rips filtration using `ripser` and find the H0
persistence diagram. The death times of H0 features record when connected components
merge. The **largest gap** between consecutive death times identifies the transition
between within-cluster distances and between-cluster distances:

$$\varepsilon^*(x) = \text{death time just before the largest gap}$$

The per-query $k_\text{eff}$ then uses the same dimension-free counting formula as
Section 18, but at scale $\varepsilon^*(x)$:

$$k_\text{eff}(x) = \frac{n_\text{maj}^\text{ball}(x, \varepsilon^*)}
                         {n_\text{min}^\text{ball}(x, \varepsilon^*)}$$

Both numerator and denominator count points within the same ball, so $\omega_d \varepsilon^{*d}$
cancels exactly — the estimate remains dimension-free.

### 27.3 Constructor

```python
KNNFairRankTopoCount(
    k_max = None,   # joint neighbourhood size; None → adaptive
    # plus all KNNFairRank parameters
)
```